# Мордовкин Максим Михайлович, Архипов И.С., N4170, Задача №3 "Многокритериальная оптимизация: метод уступок 1"

## Вариант №15
## Условие задачи

Уступка по первому критерию: 1%  
Дана двухкритериальная задача:

$$f_1 = x_1 + 2x_2 + x_3 \to \max$$
$$f_2 = 2x_1 + x_2 - x_3 \to \max$$

при ограничениях:

$$
\begin{cases}
x_1 + x_2 + x_3 \le 6\\
x_1 + x_2 \le 3\\
x_1 \le 2\\
x_1, x_2, x_3 \ge 0
\end{cases}
$$

**Метод уступок.** Сначала находим максимум $f_1$. Затем делаем уступку $\Delta = 1\%$ и максимизируем $f_2$ при дополнительном ограничении $f_1 \ge f_1^* - \Delta$.

In [ ]:
# Идея: допустимая область задачи ЛП — многогранник.
# Оптимум линейной функции всегда достигается в одной из вершин.
# Вершина — это пересечение трёх граничных плоскостей.
# Перебираем все тройки ограничений, решаем систему 3x3,
# оставляем только допустимые точки — это и есть вершины.


def det3(M):
    """Определитель матрицы 3x3 по правилу треугольников."""
    return (M[0][0] * (M[1][1] * M[2][2] - M[1][2] * M[2][1])
            - M[0][1] * (M[1][0] * M[2][2] - M[1][2] * M[2][0])
            + M[0][2] * (M[1][0] * M[2][1] - M[1][1] * M[2][0]))


def solve3(A, b):
    """Решает систему A @ x = b (3x3) по правилу Крамера.
    Возвращает (x1, x2, x3) или None, если система вырождена."""
    D = det3(A)
    if abs(D) < 1e-12:
        return None

    def replace_col(col):
        """Копия A, у которой столбец col заменён на вектор b."""
        M = [row[:] for row in A]
        for i in range(3):
            M[i][col] = b[i]
        return M

    x1 = det3(replace_col(0)) / D
    x2 = det3(replace_col(1)) / D
    x3 = det3(replace_col(2)) / D
    return (x1, x2, x3)


def is_feasible(x, constraints, eps=1e-9):
    """Проверяет, что точка x удовлетворяет всем ограничениям A_ub x <= b."""
    for (a1, a2, a3, b) in constraints:
        if a1 * x[0] + a2 * x[1] + a3 * x[2] > b + eps:
            return False
    return True


def find_vertices(constraints):
    """Находит все вершины допустимой области.

    constraints — список кортежей (a1, a2, a3, b),
    каждый задаёт неравенство a1*x1 + a2*x2 + a3*x3 <= b.
    """
    from itertools import combinations  # стандартная библиотека

    verts = []
    n = len(constraints)

    # Перебираем все тройки ограничений (как будто они активны)
    for combo in combinations(range(n), 3):
        A = []
        b = []
        for idx in combo:
            a1, a2, a3, bi = constraints[idx]
            A.append([a1, a2, a3])
            b.append(bi)
        x = solve3(A, b)
        if x is None:
            continue
        if is_feasible(x, constraints):
            # округляем, чтобы убрать дубликаты из-за погрешностей
            verts.append(tuple(round(v, 10) for v in x))

    # убираем дубликаты, сохраняя порядок
    uniq = []
    for v in verts:
        if v not in uniq:
            uniq.append(v)
    return [list(v) for v in uniq]


def evaluate(c, x):
    """Значение линейной функции c @ x."""
    return c[0] * x[0] + c[1] * x[1] + c[2] * x[2]

In [ ]:
# ШАГ 1. Максимизируем f1 = x1 + 2x2 + x3
# Каждое ограничение вида a1*x1 + a2*x2 + a3*x3 <= b
# храним как кортеж (a1, a2, a3, b).

constraints = [
    ( 1,  1,  1,  6),   #  x1 + x2 + x3 <= 6
    ( 1,  1,  0,  3),   #  x1 + x2      <= 3
    ( 1,  0,  0,  2),   #  x1           <= 2
    (-1,  0,  0,  0),   # -x1 <= 0   (x1 >= 0)
    ( 0, -1,  0,  0),   # -x2 <= 0   (x2 >= 0)
    ( 0,  0, -1,  0),   # -x3 <= 0   (x3 >= 0)
]

# Целевая функция первого критерия
c1 = (1, 2, 1)

# Находим все вершины области
verts = find_vertices(constraints)

print("Вершины допустимой области:")
for v in verts:
    print(f"  x = {v},  f1 = {evaluate(c1, v)}")

# Ищем вершину с максимальным f1
best_x1 = None
best_f1 = float("-inf")
for v in verts:
    val = evaluate(c1, v)
    if val > best_f1:
        best_f1 = val
        best_x1 = v

print("\nШАГ 1. Максимум f1")
print("x* =", best_x1)
print("f1_max =", best_f1)

Вершины допустимой области:
  x = [2.0, 1.0, 3.0],  f1 = 7.0
  x = [0.0, 3.0, 3.0],  f1 = 9.0
  x = [2.0, -0.0, 4.0],  f1 = 6.0
  x = [0.0, 0.0, 6.0],  f1 = 6.0
  x = [2.0, 1.0, 0.0],  f1 = 4.0
  x = [-0.0, 3.0, -0.0],  f1 = 6.0
  x = [2.0, 0.0, 0.0],  f1 = 2.0
  x = [-0.0, -0.0, -0.0],  f1 = -0.0

ШАГ 1. Максимум f1
x* = [0.0, 3.0, 3.0]
f1_max = 9.0


In [ ]:
# ШАГ 2. Уступка по f1 и максимизация f2 = 2x1 + x2 - x3
# По условию уступка Δ = 1% от максимума f1.
# Добавляем ограничение:
#   f1 >= f1_max - Δ   <=>   -(x1 + 2x2 + x3) <= -(f1_max - Δ)

delta = 0.01
f1_lower = best_f1 * (1 - delta)

print("ШАГ 2. Уступка по f1")
print(f"delta = {delta*100:.0f}%")
print(f"f1_lower = {f1_lower:.4f}")

# Новое ограничение: -(x1 + 2x2 + x3) <= -f1_lower
new_constraint = (-c1[0], -c1[1], -c1[2], -f1_lower)

constraints2 = constraints + [new_constraint]

# Ищем вершины расширенной области
verts2 = find_vertices(constraints2)

# Целевая функция второго критерия
c2 = (2, 1, -1)

print("\nВершины новой области и значения f2:")
for v in verts2:
    print(f"  x = {v},  f1 = {evaluate(c1, v):.4f},  f2 = {evaluate(c2, v):.4f}")

# Ищем вершину с максимальным f2
best_x2 = None
best_f2 = float("-inf")
for v in verts2:
    val = evaluate(c2, v)
    if val > best_f2 + 1e-9:   # небольшой запас на погрешность
        best_f2 = val
        best_x2 = v

print("\nОптимум по f2:")
print("x* =", best_x2)
print("f2_max =", best_f2)
print("f1 в этой точке =", evaluate(c1, best_x2))

ШАГ 2. Уступка по f1
delta = 1%
f1_lower = 8.9100

Вершины новой области и значения f2:
  x = [0.0, 3.0, 3.0],  f1 = 9.0000,  f2 = 0.0000
  x = [0.09, 2.91, 3.0],  f1 = 8.9100,  f2 = 0.0900
  x = [0.0, 2.91, 3.09],  f1 = 8.9100,  f2 = -0.1800
  x = [-0.0, 3.0, 2.91],  f1 = 8.9100,  f2 = 0.0900

Оптимум по f2:
x* = [0.09, 2.91, 3.0]
f2_max = 0.0900000000000003
f1 в этой точке = 8.91


## Пояснение

### Шаг 1. Максимизация первого критерия

Перебираем все вершины допустимой области. Значения $f_1$ в вершинах:

| Вершина | $f_1$ |
|---|---|
| $(0,0,0)$ | 0 |
| $(2,0,0)$ | 2 |
| $(0,3,0)$ | 6 |
| $(0,0,6)$ | 6 |
| $(2,1,0)$ | 4 |
| $(2,1,3)$ | 7 |
| $(2,0,4)$ | 6 |
| $(0,3,3)$ | **9** |

Максимум достигается в единственной вершине $(0,3,3)$:

$$f_1^* = 9,\qquad x^{(1)} = (0,\ 3,\ 3)$$

### Шаг 2. Уступка по первому критерию

Уступка $1\%$:

$$\Delta = 0{,}01 \cdot 9 = 0{,}09,\qquad f_1 \ge 8{,}91$$

Добавляем ограничение $x_1 + 2x_2 + x_3 \ge 8{,}91$ и снова перебираем вершины.

> **Важно: оптимум по $f_2$ не единственный.**
>
> В новой области целевая функция $f_2 = 2x_1 + x_2 - x_3$ параллельна одному из рёбер, поэтому максимум достигается не в одной точке, а на **целом отрезке** между двумя вершинами:
>
> $$x_A = (0,\ 3,\ 2{,}91),\qquad x_B = (0{,}09,\ 2{,}91,\ 3)$$
>
> В обеих точках (и во всех точках отрезка $[x_A, x_B]$):
>
> $$f_1 = 8{,}91,\qquad f_2 = 0{,}09$$
>
> Какую именно из двух вершин вернёт перебор — зависит от порядка обхода. Оба ответа корректны.

### Сводка

| Шаг | Оптимум $x$ | $f_1$ | $f_2$ |
|---|---|---|---|
| 1. Только $f_1$ | $(0,\ 3,\ 3)$ | $9$ | $-3$ |
| 2. Уступка $1\%$, макс. $f_2$ | $(0,\ 3,\ 2{,}91)$ **или** $(0{,}09,\ 2{,}91,\ 3)$ | $8{,}91$ | $0{,}09$ |

## Ответ

**Шаг 1.** Максимум первого критерия:

$$f_1^* = 9,\qquad x^{(1)} = (0,\ 3,\ 3)$$

**Шаг 2.** После уступки $\Delta = 1\%$ по $f_1$:

$$f_1 \ge 8{,}91,\qquad f_2^* = 0{,}09$$

> **Оптимум по $f_2$ не единственный.**  
> Максимум $f_2$ достигается на всём отрезке между двумя вершинами:
>
> $$x_A = (0,\ 3,\ 2{,}91),\qquad x_B = (0{,}09,\ 2{,}91,\ 3)$$
>
> В любой точке этого отрезка:
>
> $$f_1 = 8{,}91,\qquad f_2 = 0{,}09$$

**Итоговое решение по методу уступок** (одно из возможных):

$$x^* = (0,\ 3,\ 2{,}91),\qquad f_1 = 8{,}91,\qquad f_2 = 0{,}09$$

или эквивалентно:

$$x^* = (0{,}09,\ 2{,}91,\ 3),\qquad f_1 = 8{,}91,\qquad f_2 = 0{,}09$$

Уступка по первому критерию составляет $9 - 8{,}91 = 0{,}09$, то есть ровно $1\%$ от $f_1^*$.